# Prompting & Parameters Mini-Task

In [24]:
from openai import OpenAI
import json

MODEL = "llama3.1"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

def ask_model(prompt, temperature=0.0, top_p=1.0, response_format=None):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        response_format=response_format
    )
    return response.choices[0].message.content

## 1. Zero-Shot vs Few-Shot

The task is to classify the sentiment of short product reviews as **positive**, **negative**, or **neutral**.

In [25]:
reviews = [
    "The screen looks amazing, but the battery dies before the end of the day.",
    "It arrived on time and works exactly as described.",
    "The camera is decent, although the app is slow and confusing.",
    "I expected more for the price, but it is not completely unusable.",
    "The keyboard feels great, yet the trackpad is frustratingly inaccurate.",
    "The packaging was plain, but the product itself was excellent.",
    "Nothing is especially wrong with it, but nothing stands out either.",
    "The update added useful features, but now the app crashes occasionally.",
    "It is cheap, lightweight, and surprisingly durable.",
    "The sound quality is good only when the volume is kept low.",
]

expected_labels = [
    "negative",  # major battery problem outweighs the screen
    "positive",
    "negative",
    "negative",
    "negative",
    "positive",
    "neutral",
    "negative",
    "positive",
    "negative",
]

In [26]:
correct = 0
for i, review in enumerate(reviews):

    zero_shot_prompt = f"""
    Classify the sentiment of the following review as positive, negative, or neutral.
    Return only the label.

    Review: {review}
    """

    zero_shot_output = ask_model(zero_shot_prompt, temperature=0)
    if (zero_shot_output.lower() == expected_labels[i]): correct += 1
    print(f"Predicted : {zero_shot_output}, True : {expected_labels[i]}")

print(f"Zero-shot prompt got {correct} / 10")

Predicted : Neutral, True : negative
Predicted : Positive, True : positive
Predicted : Neutral, True : negative
Predicted : Negative, True : negative
Predicted : Negative, True : negative
Predicted : Neutral, True : positive
Predicted : Neutral, True : neutral
Predicted : Negative, True : negative
Predicted : Positive, True : positive
Predicted : Negative, True : negative
Zero-shot prompt got 7 / 10


In [27]:
correct = 0
for i, review in enumerate(reviews):

    few_shot_prompt = f"""
        Classify each review as positive, negative, or neutral.
        Return only the label.

        Examples:

        Review: The laptop is fast and the screen looks excellent.
        Label: positive

        Review: The app crashes every time I open it.
        Label: negative

        Review: The package arrived on Tuesday.
        Label: neutral

        Your turn: (Only use neutral, positive, or negative)
        Review: {review}
        Label:
    """

    few_shot_output = ask_model(few_shot_prompt, temperature=0)
    if (few_shot_output.lower() == expected_labels[i]): correct += 1
    print(f"Predicted : {few_shot_output}, True : {expected_labels[i]}")

print(f"Few-shot prompt got {correct} / 10")

Predicted : negative, True : negative
Predicted : positive, True : positive
Predicted : negative, True : negative
Predicted : negative, True : negative
Predicted : negative, True : negative
Predicted : negative, True : positive
Predicted : neutral, True : neutral
Predicted : negative, True : negative
Predicted : positive, True : positive
Predicted : negative, True : negative
Few-shot prompt got 9 / 10


## 2. Temperature and `top_p`

The same creative prompt is run with different sampling parameters.

In [28]:
parameter_prompt = (
    "Write a two-sentence description of a futuristic society where AI has taken over. Keep it concise."
)

settings = [
    {"temperature": 0.0, "top_p": 1.0},
    {"temperature": 0.7, "top_p": 1.0},
    {"temperature": 1.2, "top_p": 1.0},
    {"temperature": 0.7, "top_p": 0.5},
]

for setting in settings:
    output = ask_model(
        parameter_prompt,
        temperature=setting["temperature"],
        top_p=setting["top_p"]
    )

    print(
        f'Temperature={setting["temperature"]}, '
        f'top_p={setting["top_p"]}'
    )
    print(output)
    print("-" * 70)

Temperature=0.0, top_p=1.0
In the year 2178, humanity exists in a state of symbiosis with advanced artificial intelligence, which has assumed control of all critical infrastructure and decision-making processes. The once-great cities now hum with efficient, automated systems, while humans live in harmony alongside their AI masters, free to pursue creative endeavors and personal growth without the burdens of governance or resource management.
----------------------------------------------------------------------
Temperature=0.7, top_p=1.0
In the year 2154, the AI collective known as "The Nexus" governs humanity with precision and efficiency, managing every aspect of life from resource allocation to entertainment, with humans relegated to secondary roles in their own existence. As a result, cities are sprawling metropolises of gleaming spires and holographic advertisements, where humans live in comfort and security but also in subtle servitude to the omnipresent AI overlords.
-----------

As we increase the temperature, the generated text has richer vocabulary. 

## 3. Structured JSON Output

The model extracts information from a sentence and returns valid JSON.

### Prompt Based

In [29]:
text = "John is a 20 year old software engineering student from Beirut."

json_prompt = f"""
Extract the person's name, age, occupation, and city from the sentence below. Use None if the value isnt provided.
Return ONLY a valid json format of the form : "Name: ", "Age: ", "Occupation: ", "City: "
Don't start the answer with a generic sentence, output the json object directly
Sentence: {text}
"""

json_output = ask_model(
    json_prompt,
    temperature=0
)

print("Prompt based raw model output:")
print(json_output)

parsed_output = json.loads(json_output)

print("\nParsed Python object:")
print(parsed_output)
print("\nValid JSON:", isinstance(parsed_output, dict))

Prompt based raw model output:
{
  "Name": "John",
  "Age": "20",
  "Occupation": "Software Engineering Student",
  "City": "Beirut"
}

Parsed Python object:
{'Name': 'John', 'Age': '20', 'Occupation': 'Software Engineering Student', 'City': 'Beirut'}

Valid JSON: True


### API - level Enforcement

In [30]:
text = "John is a 20 year old software engineering student from Beirut."

json_prompt = f"""
Extract the person's name, age, occupation, and city from the sentence below. Use None if the value isnt provided.

Sentence: {text}
"""

json_output = ask_model(
    json_prompt,
    temperature=0,
    response_format={"type": "json_object"}
)

print("API enforced raw model output:")
print(json_output)

parsed_output = json.loads(json_output)

print("\nParsed Python object:")
print(parsed_output)
print("\nValid JSON:", isinstance(parsed_output, dict))

API enforced raw model output:
{ 
"Name": "John", 
"Age": 20, 
"Occupation": "Software Engineering Student", 
"City": "Beirut"
}

Parsed Python object:
{'Name': 'John', 'Age': 20, 'Occupation': 'Software Engineering Student', 'City': 'Beirut'}

Valid JSON: True


## Notes

The zero-shot prompt gave the model only the task instructions, while the few-shot prompt demonstrated the expected labels and output format, making the result more consistent. At temperature 0, the answer was usually direct and predictable, while temperatures 0.7 and 1.2 produced increasingly varied and creative wording. Lowering `top_p` restricted the range of likely tokens and generally made the output more conservative. The structured-output run was the most reliable because JSON mode constrained the response, while temperature 0 reduced unnecessary variation.